In [3]:
from kafi.streams.topologynode import TopologyNode as Tn

order_source_str = "orders"
sink_str = "sink"

order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn.from_records)
#
order_tn = (
    order_source_tn
    .upsert()
    .map(lambda r: r["value"])
    .map(lambda r: {"order_id": r["order_id"], "product_id": r["product_id"], "customer_id": r["customer_id"]})
)
#
self_join_group_by_tombstones_tn = (
    order_tn
    .join(
        order_tn,
        lambda l_r: l_r["customer_id"],
        lambda r_r: r_r["customer_id"],
        lambda l_r, r_r: {"product_id_1": l_r["product_id"],
                          "product_id_2": r_r["product_id"],
                          "customer_id": l_r["customer_id"]}
    )
    .filter(lambda r: r["product_id_1"] < r["product_id_2"])
    .group_by_count(
        lambda r: {"product_id_1": r["product_id_1"], "product_id_2": r["product_id_2"]},
        lambda key_r, agg_r: {"product_id_1": key_r["product_id_1"], "product_id_2": key_r["product_id_2"], "cross_purchases": agg_r}
    )
    .map(lambda r: {"value": r})
)
#
sink_tn = self_join_group_by_tombstones_tn.sink(sink_str)
#
tn = Tn.build(sink_tn)
_ = tn.from_zSet(Tn._to_records)


In [4]:
tn.reset()
#
m1 = {"key": 42,
      "value": {"order_id": 42,
                "product_id": 1,
                "customer_id": 1}}
m2 = {"key": 4711,
      "value": {"order_id": 4711,
                "product_id": 2,
                "customer_id": 1}}
print(tn.process({order_source_str: [m1, m2]}))
#
m3 = {"key": 4711,
      "value": None}
print(tn.process({order_source_str: [m3]}))
#
m4 = {"key": 4711,
      "value": {"order_id": 4711,
                "product_id": 2,
                "customer_id": 1}}
print(tn.process({order_source_str: [m4]}))


{'sink': [({'value': {'product_id_1': 1, 'product_id_2': 2, 'cross_purchases': 1}}, 1)]}
{'sink': [({'value': {'product_id_1': 1, 'product_id_2': 2, 'cross_purchases': 1}}, -1)]}
{'sink': [({'value': {'product_id_1': 1, 'product_id_2': 2, 'cross_purchases': 1}}, 1)]}
